# Beam pre-processing experiment

## Data loading

In [1]:
import numpy as np

from scripts.beamforming import filter_best_beams
from scripts.data_filter import filter_dataframe
from scripts.data_loader import load_dataframe
from scripts.utils import NETWORK_TYPE

filename = "5G_data_2023.mat"

# Series of random seeds for reproducability
random_seeds = np.loadtxt('data/random_seeds.csv', dtype=int)

# load the dataframe from saved file or 'raw' matlab file
df = load_dataframe(filename, NETWORK_TYPE._5G)

# # Drop unused columns to save space
matrix_cols_to_drop = ['toa_pps', 'toa_cir', 'toa_cov', 'campaign_id']
df['measurements_matrix'] = df['measurements_matrix'].apply(
    lambda x: x.drop(columns=matrix_cols_to_drop)
)

selected_campaigns = list(range(1, 10))
# Data filtering
df = filter_dataframe(
    df=df,
    operators=[10],
    include_columns=['pci', 'beam_index', 'nr_arfcn', 'operator_id', 'rsrq'],
    campaigns=selected_campaigns,
)

Loaded dataframe from .h5 file: /Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/data/dataframe_cache/5G_data_2023.h5


/Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/scripts/data_filter.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["measurements_matrix"] = df["measurements_matrix"].apply(


In [2]:
from scripts.utils import RF_PARAM_5G

filtered_df = df.copy()

filtered_df['measurements_matrix_cleaned'] = filtered_df['measurements_matrix'].apply(
    lambda x: filter_best_beams(x, RF_PARAM_5G.RSRQ)
)


In [6]:
diffs = []
for idx, row in filtered_df.iterrows():
    l1 = row['measurements_matrix'].shape[0]
    l2 = row['measurements_matrix_cleaned'].shape[0]
    diff = l1 - l2
    diffs.append(diff)

diffs = np.array(diffs)

print(f'mean diff {diffs.mean()}')

mean diff 49.60622942517093
